In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance


project_dir = Path.cwd()

raw_dir = project_dir / "data" / "raw"
output_dir = project_dir / "outputs" / "path_a"

results_path = raw_dir / "results.csv"
rankings_path = raw_dir / "fifa_consolidado.csv"
schedule_path = raw_dir / "world_cup_2026_schedule.csv"
odds_json_path = raw_dir / "odds_fase_grupos_copa.json"
odds_csv_path = raw_dir / "odds.csv"

output_dir.mkdir(parents=True, exist_ok=True)

print(project_dir)
print(raw_dir.exists())

d:\windows\Documents\extras\previsao_copa_26
True


In [2]:
for path in [results_path, rankings_path, schedule_path, odds_json_path]:
    print(path, path.exists())

d:\windows\Documents\extras\previsao_copa_26\data\raw\results.csv True
d:\windows\Documents\extras\previsao_copa_26\data\raw\fifa_rankings.csv True
d:\windows\Documents\extras\previsao_copa_26\data\raw\world_cup_2026_schedule.csv True
d:\windows\Documents\extras\previsao_copa_26\data\raw\odds_fase_grupos_copa.json True


In [3]:
TEAM_NAME_MAP = {
    "USA": "United States",
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Curaçao": "Curacao",
    "Côte d'Ivoire": "Ivory Coast",
    "Czech Republic": "Czechia",
    "Türkiye": "Turkey",
}


def clean_team_name(team_name):
    team_name = str(team_name).strip()
    return TEAM_NAME_MAP.get(team_name, team_name)


def extract_h2h_prices(event):
    home_team = clean_team_name(event["home_team"])
    away_team = clean_team_name(event["away_team"])

    home_prices = []
    draw_prices = []
    away_prices = []

    for bookmaker in event.get("bookmakers", []):
        for market in bookmaker.get("markets", []):
            if market.get("key") != "h2h":
                continue

            prices_by_name = {
                clean_team_name(outcome["name"]): outcome["price"]
                for outcome in market.get("outcomes", [])
            }

            if home_team in prices_by_name:
                home_prices.append(float(prices_by_name[home_team]))

            if away_team in prices_by_name:
                away_prices.append(float(prices_by_name[away_team]))

            if "Draw" in prices_by_name:
                draw_prices.append(float(prices_by_name["Draw"]))

    if not home_prices or not draw_prices or not away_prices:
        return None

    return {
        "event_date": pd.to_datetime(event["commence_time"]).date(),
        "event_home_team": home_team,
        "event_away_team": away_team,
        "event_home_win_odds": float(np.median(home_prices)),
        "event_draw_odds": float(np.median(draw_prices)),
        "event_away_win_odds": float(np.median(away_prices)),
        "n_bookmakers": len(home_prices),
    }


with odds_json_path.open("r", encoding="utf-8") as file:
    odds_events = json.load(file)

odds_rows = []

for event in odds_events:
    extracted_prices = extract_h2h_prices(event)

    if extracted_prices is not None:
        odds_rows.append(extracted_prices)

odds_data = pd.DataFrame(odds_rows)

print(odds_data.shape)
odds_data.head()

(72, 7)


,event_date,event_home_team,event_away_team,event_home_win_odds,event_draw_odds,event_away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-12,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-13,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [4]:
schedule_data = pd.read_csv(schedule_path)

schedule_data["home_team"] = schedule_data["home_team"].map(clean_team_name)
schedule_data["away_team"] = schedule_data["away_team"].map(clean_team_name)

aligned_rows = []

for _, odds_row in odds_data.iterrows():
    event_home_team = odds_row["event_home_team"]
    event_away_team = odds_row["event_away_team"]

    same_order = schedule_data[
        (schedule_data["home_team"] == event_home_team)
        & (schedule_data["away_team"] == event_away_team)
    ]

    reverse_order = schedule_data[
        (schedule_data["home_team"] == event_away_team)
        & (schedule_data["away_team"] == event_home_team)
    ]

    if not same_order.empty:
        schedule_row = same_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_home_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_away_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    elif not reverse_order.empty:
        schedule_row = reverse_order.iloc[0]

        aligned_rows.append(
            {
                "match_date": schedule_row["match_date"],
                "home_team": schedule_row["home_team"],
                "away_team": schedule_row["away_team"],
                "home_win_odds": odds_row["event_away_win_odds"],
                "draw_odds": odds_row["event_draw_odds"],
                "away_win_odds": odds_row["event_home_win_odds"],
                "n_bookmakers": odds_row["n_bookmakers"],
            }
        )

    else:
        print("Não encontrei na tabela:", event_home_team, "x", event_away_team)

odds_aligned_data = pd.DataFrame(aligned_rows)

odds_aligned_data.to_csv(odds_csv_path, index=False)

print(odds_aligned_data.shape)
odds_aligned_data.head()

(72, 7)


,match_date,home_team,away_team,home_win_odds,draw_odds,away_win_odds,n_bookmakers
0,2026-06-11,Mexico,South Africa,1.41,4.50,8.725,28
1,2026-06-11,South Korea,Czechia,2.70,3.10,2.850,27
2,2026-06-12,Canada,Bosnia and Herzegovina,1.81,3.60,4.700,27
3,2026-06-12,United States,Paraguay,1.94,3.40,4.200,27
4,2026-06-13,Qatar,Switzerland,13.25,6.75,1.200,27


In [5]:
import poison_26 as path_a

In [6]:
historical_matches_raw = pd.read_csv(results_path)
rankings_raw = pd.read_csv(rankings_path)
schedule_raw = pd.read_csv(schedule_path)
odds_raw = pd.read_csv(odds_csv_path)

historical_matches_raw = historical_matches_raw.copy()

score_columns = ["home_score", "away_score"]

historical_matches_raw["date"] = pd.to_datetime(
    historical_matches_raw["date"],
    errors="coerce",
)

historical_matches_raw["home_score"] = pd.to_numeric(
    historical_matches_raw["home_score"],
    errors="coerce",
)

historical_matches_raw["away_score"] = pd.to_numeric(
    historical_matches_raw["away_score"],
    errors="coerce",
)

missing_score_data = historical_matches_raw[
    historical_matches_raw[score_columns].isna().any(axis=1)
].copy()

print("Jogos sem placar:", missing_score_data.shape[0])

display(
    missing_score_data[
        ["date", "home_team", "away_team", "home_score", "away_score", "tournament"]
    ].tail(20)
)

historical_matches_clean = historical_matches_raw.dropna(
    subset=["date", "home_score", "away_score"]
).copy()

historical_matches_clean = historical_matches_clean[
    historical_matches_clean["date"] < pd.Timestamp("2026-06-11")
].copy()

historical_matches_clean["home_score"] = historical_matches_clean["home_score"].astype(int)
historical_matches_clean["away_score"] = historical_matches_clean["away_score"].astype(int)

historical_matches = path_a.standardize_matches_columns(historical_matches_clean)
schedule_data = path_a.standardize_schedule_columns(schedule_raw)

ranking_lookup = path_a.RankingLookup(rankings_raw)
odds_lookup = path_a.OddsLookup(odds_raw)

print("Histórico bruto:", historical_matches_raw.shape)
print("Histórico limpo:", historical_matches.shape)
print("Tabela Copa:", schedule_data.shape)
print("Odds:", odds_raw.shape)

Jogos sem placar: 72


,date,home_team,away_team,home_score,away_score,tournament
49452,2026-06-24,Scotland,Brazil,NaN,NaN,FIFA World Cup
49453,2026-06-24,Morocco,Haiti,NaN,NaN,FIFA World Cup
49454,2026-06-25,United States,Turkey,NaN,NaN,FIFA World Cup
49455,2026-06-25,Paraguay,Australia,NaN,NaN,FIFA World Cup
49456,2026-06-25,Curaçao,Ivory Coast,NaN,NaN,FIFA World Cup
49457,2026-06-25,Ecuador,Germany,NaN,NaN,FIFA World Cup
49458,2026-06-25,Japan,Sweden,NaN,NaN,FIFA World Cup
49459,2026-06-25,Tunisia,Netherlands,NaN,NaN,FIFA World Cup
49460,2026-06-26,Egypt,Iran,NaN,NaN,FIFA World Cup
49461,2026-06-26,New Zealand,Belgium,NaN,NaN,FIFA World Cup


Histórico bruto: (49472, 9)
Histórico limpo: (49400, 9)
Tabela Copa: (72, 8)
Odds: (72, 7)


In [7]:
market_features = [
    "market_home_prob",
    "market_draw_prob",
    "market_away_prob",
]

path_a.FEATURE_COLUMNS = [
    feature
    for feature in path_a.FEATURE_COLUMNS
    if feature not in market_features
]

odds_lookup_for_training = path_a.OddsLookup(None)

features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup_for_training,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(features_data.shape)
print(path_a.FEATURE_COLUMNS)

(49400, 23)
['elo_diff', 'attack_diff', 'defense_diff', 'form_diff', 'home_rank', 'away_rank', 'rank_diff', 'home_total_points', 'away_total_points', 'fifa_points_diff', 'neutral', 'is_non_neutral', 'tournament_weight', 'home_attack', 'away_attack', 'home_defense', 'away_defense', 'home_form', 'away_form', 'home_decay', 'away_decay', 'home_matches_played', 'away_matches_played']


In [8]:
features_data, home_goals, away_goals, initial_states = path_a.build_training_data(
    historical_matches=historical_matches,
    ranking_lookup=ranking_lookup,
    odds_lookup=odds_lookup,
)

home_model = path_a.train_poisson_model(features_data, home_goals)
away_model = path_a.train_poisson_model(features_data, away_goals)

print(home_goals.shape)
print(away_goals.shape)

(49400,)
(49400,)


In [9]:
def extract_poisson_weights(model, model_name):
    scaler = model.named_steps["scaler"]
    poisson = model.named_steps["poisson"]

    standardized_coefficients = pd.Series(
        poisson.coef_,
        index=path_a.FEATURE_COLUMNS,
        name="standardized_coefficient",
    )

    raw_scale_coefficients = standardized_coefficients / scaler.scale_

    weights_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "standardized_coefficient": standardized_coefficients.values,
            "raw_scale_coefficient": raw_scale_coefficients.values,
            "abs_standardized_coefficient": standardized_coefficients.abs().values,
        }
    )

    weights_data["model"] = model_name

    return weights_data.sort_values(
        "abs_standardized_coefficient",
        ascending=False,
    ).reset_index(drop=True)


home_weights = extract_poisson_weights(home_model, "home_goals")
away_weights = extract_poisson_weights(away_model, "away_goals")

display(home_weights.head(30))
display(away_weights.head(30))

,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,0.363339,0.001551,0.363339,home_goals
1,away_defense,0.112290,0.151353,0.112290,home_goals
2,away_matches_played,-0.106998,-0.000479,0.106998,home_goals
3,form_diff,-0.106458,-0.508431,0.106458,home_goals
4,home_attack,0.093052,0.154153,0.093052,home_goals
5,defense_diff,0.085129,0.101248,0.085129,home_goals
6,away_form,0.079944,0.500883,0.079944,home_goals
7,attack_diff,0.063711,0.089458,0.063711,home_goals
8,home_form,-0.059550,-0.372061,0.059550,home_goals
9,away_decay,-0.044939,-0.280950,0.044939,home_goals


,feature,standardized_coefficient,raw_scale_coefficient,abs_standardized_coefficient,model
0,elo_diff,-0.366300,-0.001564,0.366300,away_goals
1,home_defense,0.129035,0.183949,0.129035,away_goals
2,away_attack,0.116203,0.194508,0.116203,away_goals
3,form_diff,0.099621,0.475776,0.099621,away_goals
4,defense_diff,-0.090978,-0.108205,0.090978,away_goals
5,away_form,-0.078540,-0.492085,0.078540,away_goals
6,home_matches_played,-0.059225,-0.000255,0.059225,away_goals
7,home_attack,0.054622,0.090489,0.054622,away_goals
8,home_form,0.052006,0.324923,0.052006,away_goals
9,attack_diff,-0.051181,-0.071864,0.051181,away_goals


In [10]:
def calculate_permutation_importance(model, features_data, target, model_name):
    result = permutation_importance(
        model,
        features_data[path_a.FEATURE_COLUMNS],
        target,
        n_repeats=10,
        random_state=42,
        scoring="neg_mean_absolute_error",
    )

    importance_data = pd.DataFrame(
        {
            "feature": path_a.FEATURE_COLUMNS,
            "importance_mean": result.importances_mean,
            "importance_std": result.importances_std,
            "model": model_name,
        }
    )

    return importance_data.sort_values(
        "importance_mean",
        ascending=False,
    ).reset_index(drop=True)


home_importance = calculate_permutation_importance(
    home_model,
    features_data,
    home_goals,
    "home_goals",
)

away_importance = calculate_permutation_importance(
    away_model,
    features_data,
    away_goals,
    "away_goals",
)

display(home_importance.head(30))
display(away_importance.head(30))

,feature,importance_mean,importance_std,model
0,elo_diff,2.094460e-01,0.001271,home_goals
1,form_diff,4.058766e-02,0.001118,home_goals
2,away_defense,2.465366e-02,0.000757,home_goals
3,away_form,2.323788e-02,0.000855,home_goals
4,home_attack,1.309773e-02,0.000523,home_goals
5,away_matches_played,1.120174e-02,0.000948,home_goals
6,home_form,1.026813e-02,0.000519,home_goals
7,defense_diff,9.817180e-03,0.000509,home_goals
8,attack_diff,2.727204e-03,0.000543,home_goals
9,away_decay,2.112383e-03,0.000416,home_goals


,feature,importance_mean,importance_std,model
0,elo_diff,0.121183,0.002131,away_goals
1,form_diff,0.023610,0.000556,away_goals
2,home_defense,0.016730,0.000622,away_goals
3,away_form,0.011656,0.000401,away_goals
4,away_attack,0.009581,0.000611,away_goals
5,home_form,0.007384,0.000342,away_goals
6,home_attack,0.005578,0.000384,away_goals
7,defense_diff,0.004639,0.000446,away_goals
8,neutral,0.002358,0.000342,away_goals
9,is_non_neutral,0.002358,0.000342,away_goals


In [11]:
MARKET_WEIGHT = 0.40
MAX_GOALS = 10


def calculate_poisson_probabilities(goal_rate, max_goals=MAX_GOALS):
    probabilities = np.zeros(max_goals + 1)
    probabilities[0] = np.exp(-goal_rate)

    for goals in range(1, max_goals + 1):
        probabilities[goals] = probabilities[goals - 1] * goal_rate / goals

    probabilities = probabilities / probabilities.sum()

    return probabilities


def calculate_poisson_outcome_probabilities(
    home_lambda,
    away_lambda,
    max_goals=MAX_GOALS,
):
    home_probabilities = calculate_poisson_probabilities(home_lambda, max_goals)
    away_probabilities = calculate_poisson_probabilities(away_lambda, max_goals)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    home_win_probability = np.tril(score_matrix, k=-1).sum()
    draw_probability = np.trace(score_matrix)
    away_win_probability = np.triu(score_matrix, k=1).sum()

    total_probability = (
        home_win_probability
        + draw_probability
        + away_win_probability
    )

    return {
        "home_win": home_win_probability / total_probability,
        "draw": draw_probability / total_probability,
        "away_win": away_win_probability / total_probability,
    }


def normalize_probabilities(probabilities):
    total_probability = sum(probabilities.values())

    if total_probability <= 0:
        raise ValueError("A soma das probabilidades precisa ser positiva.")

    return {
        key: value / total_probability
        for key, value in probabilities.items()
    }


def market_probabilities_to_outcome_probabilities(market_probabilities):
    return normalize_probabilities(
        {
            "home_win": market_probabilities["market_home_prob"],
            "draw": market_probabilities["market_draw_prob"],
            "away_win": market_probabilities["market_away_prob"],
        }
    )


def has_real_market_probabilities(market_probabilities):
    default_probabilities = path_a.DEFAULT_MARKET_PROBABILITIES

    return any(
        abs(market_probabilities[key] - default_probabilities[key]) > 1e-10
        for key in default_probabilities
    )


def blend_outcome_probabilities(
    model_probabilities,
    market_probabilities,
    market_weight=MARKET_WEIGHT,
):
    if not has_real_market_probabilities(market_probabilities):
        return normalize_probabilities(model_probabilities)

    market_outcome_probabilities = market_probabilities_to_outcome_probabilities(
        market_probabilities
    )

    blended_probabilities = {
        outcome: (
            (1.0 - market_weight) * model_probabilities[outcome]
            + market_weight * market_outcome_probabilities[outcome]
        )
        for outcome in ["home_win", "draw", "away_win"]
    }

    return normalize_probabilities(blended_probabilities)

In [12]:
def get_score_outcome(home_goals, away_goals):
    if home_goals > away_goals:
        return "home_win"

    if home_goals == away_goals:
        return "draw"

    return "away_win"


def sample_score_from_blended_probabilities(
    home_lambda,
    away_lambda,
    target_probabilities,
    random_generator,
    max_goals=MAX_GOALS,
):
    home_probabilities = calculate_poisson_probabilities(home_lambda, max_goals)
    away_probabilities = calculate_poisson_probabilities(away_lambda, max_goals)

    score_matrix = np.outer(home_probabilities, away_probabilities)

    model_outcome_probabilities = {
        "home_win": 0.0,
        "draw": 0.0,
        "away_win": 0.0,
    }

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            outcome = get_score_outcome(home_goals, away_goals)
            model_outcome_probabilities[outcome] += score_matrix[
                home_goals,
                away_goals,
            ]

    adjusted_score_matrix = score_matrix.copy()

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):
            outcome = get_score_outcome(home_goals, away_goals)
            model_probability = model_outcome_probabilities[outcome]

            if model_probability > 0:
                adjustment_factor = (
                    target_probabilities[outcome] / model_probability
                )
                adjusted_score_matrix[home_goals, away_goals] *= adjustment_factor

    adjusted_score_matrix = adjusted_score_matrix / adjusted_score_matrix.sum()

    flat_probabilities = adjusted_score_matrix.ravel()
    selected_index = random_generator.choice(
        len(flat_probabilities),
        p=flat_probabilities,
    )

    home_goals, away_goals = np.unravel_index(
        selected_index,
        adjusted_score_matrix.shape,
    )

    return int(home_goals), int(away_goals)

In [13]:
def simulate_match_with_market_blend(
    home_team,
    away_team,
    match_date,
    neutral,
    tournament,
    stage,
    group,
    states,
    ranking_lookup,
    odds_lookup,
    home_model,
    away_model,
    random_generator,
    knockout=False,
):
    market_probabilities = odds_lookup.get(
        match_date,
        home_team,
        away_team,
    )

    feature_row = path_a.build_feature_row(
        home_team=home_team,
        away_team=away_team,
        match_date=match_date,
        neutral=neutral,
        tournament=tournament,
        states=states,
        ranking_lookup=ranking_lookup,
        market_probabilities=market_probabilities,
    )

    home_lambda, away_lambda = path_a.predict_expected_goals(
        home_model,
        away_model,
        feature_row,
    )

    model_probabilities = calculate_poisson_outcome_probabilities(
        home_lambda,
        away_lambda,
    )

    target_probabilities = blend_outcome_probabilities(
        model_probabilities=model_probabilities,
        market_probabilities=market_probabilities,
        market_weight=MARKET_WEIGHT,
    )

    home_goals, away_goals = sample_score_from_blended_probabilities(
        home_lambda=home_lambda,
        away_lambda=away_lambda,
        target_probabilities=target_probabilities,
        random_generator=random_generator,
    )

    winner = path_a.infer_winner_from_score(
        home_team,
        away_team,
        home_goals,
        away_goals,
    )

    home_result_override = None

    if knockout and winner is None:
        home_advancement_probability = (
            target_probabilities["home_win"]
            / (
                target_probabilities["home_win"]
                + target_probabilities["away_win"]
            )
        )

        home_advancement_probability = float(
            np.clip(home_advancement_probability, 0.35, 0.65)
        )

        if random_generator.random() < home_advancement_probability:
            winner = home_team
            home_result_override = 0.55
        else:
            winner = away_team
            home_result_override = 0.45

    path_a.update_team_states(
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        match_date=match_date,
        tournament=tournament,
        states=states,
        home_result_override=home_result_override,
    )

    return path_a.MatchSimulation(
        home_team=home_team,
        away_team=away_team,
        home_goals=home_goals,
        away_goals=away_goals,
        winner=winner,
        stage=stage,
        group=group,
    )


path_a.simulate_match = simulate_match_with_market_blend

In [20]:
def run_model_with_market_weight(market_weight, simulations=5000):
    global MARKET_WEIGHT

    MARKET_WEIGHT = market_weight

    results = path_a.run_simulations(
        schedule_data=schedule_data,
        playoff_data=None,
        initial_states=initial_states,
        ranking_lookup=ranking_lookup,
        odds_lookup=odds_lookup,
        home_model=home_model,
        away_model=away_model,
        simulations=simulations,
        seed=42,
    )

    champion_data = pd.DataFrame(
        [
            {
                "team": team,
                "count": count,
                "champion_probability": count / simulations,
                "market_weight": market_weight,
            }
            for team, count in results["champion"].most_common()
        ]
    )

    return champion_data


comparison_data = pd.concat(
    [
        run_model_with_market_weight(0.00, simulations=1000),
        run_model_with_market_weight(0.20, simulations=1000),
        run_model_with_market_weight(0.35, simulations=1000),
        run_model_with_market_weight(0.40, simulations=1000),
        run_model_with_market_weight(0.50, simulations=1000),
        run_model_with_market_weight(0.60, simulations=1000),
        run_model_with_market_weight(0.75, simulations=1000),
    ],
    ignore_index=True,
)

comparison_data.head(30)

,team,count,champion_probability,market_weight
0,Spain,236,0.236,0.0
1,Argentina,110,0.110,0.0
2,France,87,0.087,0.0
3,Germany,76,0.076,0.0
4,Brazil,71,0.071,0.0
5,England,48,0.048,0.0
6,Colombia,36,0.036,0.0
7,Netherlands,34,0.034,0.0
8,Portugal,33,0.033,0.0
9,Belgium,32,0.032,0.0


In [21]:
comparison_pivot = comparison_data.pivot_table(
    index="team",
    columns="market_weight",
    values="champion_probability",
    fill_value=0,
)

comparison_pivot["max_probability"] = comparison_pivot.max(axis=1)

comparison_pivot = comparison_pivot.sort_values(
    "max_probability",
    ascending=False,
)

comparison_pivot.head(30)

market_weight,0.0,0.2,0.35,0.4,0.5,0.6,0.75,max_probability
team,,,,,,,,
Spain,0.236,0.235,0.260,0.218,0.262,0.234,0.222,0.262
Argentina,0.110,0.112,0.116,0.125,0.090,0.136,0.118,0.136
Brazil,0.071,0.083,0.073,0.092,0.087,0.079,0.083,0.092
France,0.087,0.089,0.076,0.082,0.078,0.089,0.087,0.089
Germany,0.076,0.068,0.068,0.076,0.060,0.054,0.064,0.076
England,0.048,0.062,0.054,0.065,0.056,0.062,0.061,0.065
Portugal,0.033,0.033,0.023,0.039,0.044,0.027,0.042,0.044
Uruguay,0.031,0.032,0.043,0.021,0.030,0.028,0.031,0.043
Netherlands,0.034,0.035,0.030,0.041,0.031,0.039,0.035,0.041
